# Running Through Windows: static word embeddings shift sweep

This notebook uses the smoother shift-window method from the original second method cell: the spike window stays at each word duration (`onset_to_offset`) and we sweep `shift_ms` across time.

For the five added patients, the static word embeddings are already cached in `ConvoDATAS/EmbedCache` as `fasttext-wiki` arrays with shape `(1, n_words, 300)`. The setup cells below convert those cached arrays plus the transcript Excel files into the older `*_words_word2vec` CSV/XLSX layout expected by the original window-sweep helper.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

PROJECT_DIR = Path("/scratch/aniluchavez/hippocampal-speaker-semantics")
CONVO_DIR = Path("/scratch/aniluchavez/ConvoDATAS")
TRANSCRIPTS_DIR = CONVO_DIR / "Transcripts"
SPIKES_MAT_ROOT = CONVO_DIR / "SpikesMAT"
EMBED_CACHE_DIR = CONVO_DIR / "EmbedCache"

MODEL_TAG = "fasttext-wiki"   # static 300-d word embedding cache
EMBED_LAYER = 0                # static embeddings are saved as (1, n_words, 300)

# Generated compatibility layout for the old word2vec pipeline functions.
WORD2VEC_ROOT = CONVO_DIR / "Word2VecFromEmbedCache_fasttext-wiki"
SPIKE_OUTPUT_ROOT = CONVO_DIR / "spikesforw2v_shift_windows_fasttext-wiki"
RESULTS_DIR = PROJECT_DIR / "results" / "runningthroughwindows_fasttext_wiki"

WORD2VEC_ROOT.mkdir(parents=True, exist_ok=True)
SPIKE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

for module_dir in (PROJECT_DIR / "scripts", PROJECT_DIR / "utils"):
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

from run_spike_pipeline import run_patient_pipeline
from build_XY import build_cleanX_from_spike_dict, clean_inputs
from sweep_regression import sweep_all_configs

print(f"Project: {PROJECT_DIR}")
print(f"Transcripts: {TRANSCRIPTS_DIR}")
print(f"Embedding cache: {EMBED_CACHE_DIR}")
print(f"Generated word2vec-style inputs: {WORD2VEC_ROOT}")
print(f"Spikes MAT root: {SPIKES_MAT_ROOT}")
print(f"Spike outputs: {SPIKE_OUTPUT_ROOT}")
print(f"Regression outputs: {RESULTS_DIR}")

## Added Patients

In [ ]:
NEW_PATIENT_CONFIGS = [
    {
        "patient_id": "ptYFM_task104",
        "patient_prefix": "PTYFM_task104",
        "region_ranges": {"hippocampus": [(33, 48)]},
    },
    {
        "patient_id": "ptYFP_task88",
        "patient_prefix": "PTYFP_task88",
        "region_ranges": {"hippocampus": [(17, 24), (25, 32), (49, 56), (57, 64)]},
    },
    {
        "patient_id": "ptYFR_task91",
        "patient_prefix": "PTYFR_task91",
        "region_ranges": {"hippocampus": [(1, 16), (41, 56)]},
    },
    {
        "patient_id": "ptYFS_task95",
        "patient_prefix": "PTYFS_task95",
        "region_ranges": {"hippocampus": [(1, 24)]},
    },
    {
        "patient_id": "ptYFU_task224",
        "patient_prefix": "PTYFU_task224",
        "region_ranges": {"hippocampus": [(17, 32), (41, 56)]},
    },
]

alpha_grid_final = np.unique(np.concatenate([
    [0.1],
    np.logspace(0, 2.5, 10),
    [1000.0],
    np.logspace(-2, 4, 20),
]))

NEW_PATIENT_CONFIGS

## Build word2vec-style inputs from `EmbedCache`

The old pipeline expects per-patient folders containing:

- `{patient}_aligned_word2vec_embeddings.csv` with `Word`, `Speaker`, `RowIndex`, `Embedding`
- `{patient}_filtered_used_rows_word2vec.xlsx` with the word timing and speaker columns

This cell creates those files from `Transcripts` plus the cached `fasttext-wiki` arrays.

In [ ]:
def transcript_path(patient_prefix):
    exact = TRANSCRIPTS_DIR / f"{patient_prefix}_filtered_used_rows_withNP_withClusterIDNew.xlsx"
    if exact.exists():
        return exact
    matches = sorted(TRANSCRIPTS_DIR.glob(f"{patient_prefix}_filtered_used_rows_withNP_withClusterID*.xlsx"))
    matches = [p for p in matches if not p.name.startswith(("._", "~$"))]
    if not matches:
        raise FileNotFoundError(f"No transcript Excel found for {patient_prefix} in {TRANSCRIPTS_DIR}")
    return matches[-1]


def embedding_cache_path(patient_prefix):
    return EMBED_CACHE_DIR / f"{patient_prefix}_{MODEL_TAG}_word_emb_layers.npy"


def word2vec_paths(patient_prefix, word2vec_root=WORD2VEC_ROOT):
    folder = Path(word2vec_root) / f"{patient_prefix}_words_word2vec"
    return {
        "word2vec_folder": folder,
        "embedding_csv": folder / f"{patient_prefix}_aligned_word2vec_embeddings.csv",
        "word2vec_excel": folder / f"{patient_prefix}_filtered_used_rows_word2vec.xlsx",
    }


def _speaker_columns(df):
    cols = [c for c in df.columns if str(c).lower().startswith("speaker")]
    def key(col):
        digits = "".join(ch for ch in str(col) if ch.isdigit())
        return int(digits) if digits else 999
    return sorted(cols, key=key)


def _word_and_speaker_from_row(row, speaker_cols):
    for col in speaker_cols:
        value = row.get(col)
        if pd.notna(value) and str(value).strip() not in ("", "nan", "xxx"):
            digits = "".join(ch for ch in str(col) if ch.isdigit())
            return str(value).strip(), f"SPK{int(digits)}"
    fallback = row.get("CleanedWord", row.get("AllWords", ""))
    return str(fallback).strip(), ""


def prepare_word2vec_compatible_inputs(patient_configs, overwrite=False):
    rows = []
    for cfg in patient_configs:
        patient_prefix = cfg["patient_prefix"]
        xlsx_in = transcript_path(patient_prefix)
        npy_in = embedding_cache_path(patient_prefix)
        paths = word2vec_paths(patient_prefix)
        paths["word2vec_folder"].mkdir(parents=True, exist_ok=True)

        if not npy_in.exists():
            raise FileNotFoundError(f"Missing embedding cache: {npy_in}")

        if paths["embedding_csv"].exists() and paths["word2vec_excel"].exists() and not overwrite:
            status = "exists"
        else:
            df = pd.read_excel(xlsx_in)
            df["onset"] = pd.to_numeric(df["onset"], errors="coerce")
            df = df.dropna(subset=["onset"]).sort_values("onset").reset_index(drop=True)

            emb = np.load(npy_in, mmap_mode="r")[EMBED_LAYER].astype(np.float32)
            if emb.shape[0] != len(df):
                raise ValueError(
                    f"{patient_prefix}: embedding rows ({emb.shape[0]}) != transcript rows ({len(df)})"
                )

            speaker_cols = _speaker_columns(df)
            words, speakers = zip(*[_word_and_speaker_from_row(row, speaker_cols) for _, row in df.iterrows()])

            meta = pd.DataFrame({
                "Word": list(words),
                "Speaker": list(speakers),
                "RowIndex": np.arange(len(df), dtype=int),
                "Embedding": [np.asarray(vec, dtype=float).tolist() for vec in emb],
            })
            meta.to_csv(paths["embedding_csv"], index=False)
            df.to_excel(paths["word2vec_excel"], index=False)
            status = "created"

        rows.append({
            "patient_prefix": patient_prefix,
            "status": status,
            "transcript": str(xlsx_in),
            "embedding_cache": str(npy_in),
            "embedding_csv": str(paths["embedding_csv"]),
            "word2vec_excel": str(paths["word2vec_excel"]),
        })
    return pd.DataFrame(rows)

prepared_inputs = prepare_word2vec_compatible_inputs(NEW_PATIENT_CONFIGS, overwrite=False)
prepared_inputs

## Input Checks

In [ ]:
def patient_spikes_dir(patient_id):
    subfolder = patient_id.split("_")[0].replace("pt", "")
    return SPIKES_MAT_ROOT / subfolder


def preflight_patient_inputs(patient_configs, word2vec_root=WORD2VEC_ROOT):
    rows = []
    for cfg in patient_configs:
        patient_id = cfg["patient_id"]
        patient_prefix = cfg["patient_prefix"]
        paths = word2vec_paths(patient_prefix, word2vec_root)
        mat_base = patient_spikes_dir(patient_id)
        mat_file = mat_base / f"{patient_id}_new_spikes.mat"
        cache_file = embedding_cache_path(patient_prefix)
        row = {
            "patient_prefix": patient_prefix,
            "mat_file_exists": mat_file.exists(),
            "embedding_cache_exists": cache_file.exists(),
            "embedding_csv_exists": paths["embedding_csv"].exists(),
            "word2vec_excel_exists": paths["word2vec_excel"].exists(),
            "mat_file": str(mat_file),
            "embedding_cache": str(cache_file),
            "embedding_csv": str(paths["embedding_csv"]),
            "word2vec_excel": str(paths["word2vec_excel"]),
        }
        row["ready"] = row["mat_file_exists"] and row["embedding_cache_exists"] and row["embedding_csv_exists"] and row["word2vec_excel_exists"]
        rows.append(row)
    return pd.DataFrame(rows)

preflight = preflight_patient_inputs(NEW_PATIENT_CONFIGS)
preflight

## Shift-Window Method

This is the original smoother method, updated to use scratch paths and generated word2vec-style inputs. Each shift calls `run_patient_pipeline(..., use_sliding_window=True, shift_ms=shift)` with `onset_to_offset` windows.

In [ ]:
def _raise_for_missing_inputs(patient_id, patient_prefix, word2vec_root=WORD2VEC_ROOT):
    mat_file = patient_spikes_dir(patient_id) / f"{patient_id}_new_spikes.mat"
    paths = word2vec_paths(patient_prefix, word2vec_root)
    required = [mat_file, embedding_cache_path(patient_prefix), paths["embedding_csv"], paths["word2vec_excel"]]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        msg = "Missing required input files:\n" + "\n".join(f"  - {p}" for p in missing)
        raise FileNotFoundError(msg)
    return mat_file, paths


def sweep_shifts_over_patient(patient_id, patient_prefix,
                              regions=("hippocampus",),
                              region_ranges=None,
                              shift_range=range(-1000, 1001, 50),
                              test_sizes=(0.3,),
                              n_pcs_list=(30,),
                              alpha_grid=(0.01, 0.1, 1.0),
                              n_shuffles=20,
                              n_jobs=4,
                              word2vec_root=WORD2VEC_ROOT,
                              spike_base_dir=SPIKE_OUTPUT_ROOT,
                              results_dir=RESULTS_DIR):
    mat_file, paths = _raise_for_missing_inputs(patient_id, patient_prefix, word2vec_root)
    mat_base = mat_file.parent
    embedding_csv = paths["embedding_csv"]
    base_excel = paths["word2vec_excel"]

    df_words = pd.read_excel(base_excel)
    speaker_columns = [
        col for col in df_words.columns
        if str(col).lower().startswith("speaker") and df_words[col].notna().any()
    ]
    if not speaker_columns:
        raise ValueError(f"No non-empty Speaker* columns found in {base_excel}")

    print(f"Found speaker columns for {patient_prefix}: {speaker_columns}")
    window_sweep_config = {spk: {"mode": "onset_to_offset"} for spk in speaker_columns}

    all_results = []
    for shift in shift_range:
        print(f"\n=== Running {patient_prefix} shift {shift:+} ms ===")
        output_suffix = f"shift{shift:+d}ms".replace("+", "p").replace("-", "m")

        spike_data, spike_dir, meta_xlsx = run_patient_pipeline(
            patient_id=patient_id,
            patient_prefix=patient_prefix,
            mat_base=str(mat_base),
            excel_base=str(word2vec_root),
            region_ranges=region_ranges,
            speaker_window_modes=window_sweep_config,
            output_suffix=output_suffix,
            spike_base_dir=str(spike_base_dir),
            use_sliding_window=True,
            shift_ms=shift,
            return_spike_data=True,
        )

        for region in regions:
            print(f"Region: {region}")
            (X_self, Y_self, dur_self), (X_other, Y_other, dur_other) = build_cleanX_from_spike_dict(
                str(embedding_csv), meta_xlsx, spike_data,
                region=region,
                target_speaker="SPK1",
                embedding_col="Embedding",
                separate_self_other=True,
            )

            X_self, Y_self, dur_self = clean_inputs(X_self, Y_self, dur_self)
            X_other, Y_other, dur_other = clean_inputs(X_other, Y_other, dur_other)

            def run_condition(X, Y, dur, label):
                if len(X) == 0 or Y.ndim < 2 or Y.shape[1] == 0:
                    print(f"Skipping {label}: X={getattr(X, 'shape', None)}, Y={getattr(Y, 'shape', None)}")
                    return pd.DataFrame()
                out = sweep_all_configs(
                    X_embed=X,
                    dur=dur,
                    Y=Y,
                    test_sizes=list(test_sizes),
                    n_pcs_list=list(n_pcs_list),
                    alpha_grid=np.asarray(alpha_grid),
                    n_shuffles=n_shuffles,
                    n_jobs=n_jobs,
                )
                out["patient_prefix"] = patient_prefix
                out["model_tag"] = MODEL_TAG
                out["condition"] = label
                out["shift_ms"] = shift
                out["region"] = region
                return out

            all_results.append(run_condition(X_self, Y_self, dur_self, "self"))
            all_results.append(run_condition(X_other, Y_other, dur_other, "other"))

    nonempty = [df for df in all_results if not df.empty]
    df_all = pd.concat(nonempty, ignore_index=True) if nonempty else pd.DataFrame()
    output_csv = Path(results_dir) / f"regression_results_{patient_prefix}_{MODEL_TAG}_ALLREGIONS_shifts.csv"
    df_all.to_csv(output_csv, index=False)
    print(f"Saved all-region shift sweep to: {output_csv}")
    return df_all


def run_shift_sweep_all_patients(patient_configs,
                                 shift_range=range(-1000, 1001, 10),
                                 test_sizes=(None,),
                                 n_pcs_list=(100,),
                                 alpha_grid=alpha_grid_final,
                                 n_shuffles=20,
                                 n_jobs=-1,
                                 word2vec_root=WORD2VEC_ROOT,
                                 spike_base_dir=SPIKE_OUTPUT_ROOT,
                                 results_dir=RESULTS_DIR):
    all_patient_results = []
    for cfg in patient_configs:
        patient_id = cfg["patient_id"]
        patient_prefix = cfg["patient_prefix"]
        region_map = cfg["region_ranges"]
        regions = list(region_map.keys())
        print(f"\nRunning patient: {patient_prefix} | Regions: {regions}")
        result = sweep_shifts_over_patient(
            patient_id=patient_id,
            patient_prefix=patient_prefix,
            regions=regions,
            region_ranges=region_map,
            shift_range=shift_range,
            test_sizes=test_sizes,
            n_pcs_list=n_pcs_list,
            alpha_grid=alpha_grid,
            n_shuffles=n_shuffles,
            n_jobs=n_jobs,
            word2vec_root=word2vec_root,
            spike_base_dir=spike_base_dir,
            results_dir=results_dir,
        )
        all_patient_results.append(result)
    return all_patient_results

## Run

Leave `RUN_SWEEP = False` while checking paths. Set it to `True` to launch the five-patient shift sweep.

In [ ]:
RUN_SWEEP = False

if RUN_SWEEP:
    results = run_shift_sweep_all_patients(
        patient_configs=NEW_PATIENT_CONFIGS,
        shift_range=range(-1000, 1001, 10),
        test_sizes=(None,),
        n_pcs_list=(100,),
        alpha_grid=alpha_grid_final,
        n_shuffles=20,
        n_jobs=-1,
    )
else:
    missing = preflight.loc[~preflight["ready"], ["patient_prefix", "mat_file_exists", "embedding_cache_exists", "embedding_csv_exists", "word2vec_excel_exists"]]
    if missing.empty:
        print("All inputs found/generated. Set RUN_SWEEP = True to launch the five-patient shift sweep.")
    else:
        print("Not launching sweep yet. Missing inputs:")
        display(missing)

## Quick Plot After Results Exist

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

result_files = sorted(RESULTS_DIR.glob(f"regression_results_*_{MODEL_TAG}_ALLREGIONS_shifts.csv"))
if result_files:
    df = pd.concat([pd.read_csv(p) for p in result_files], ignore_index=True)
    for (patient, region), df_region in df.groupby(["patient_prefix", "region"]):
        plt.figure(figsize=(10, 5))
        sns.lineplot(
            data=df_region,
            x="shift_ms",
            y="ll_diff",
            hue="condition",
            errorbar="ci",
            marker="o",
        )
        plt.axvline(0, color="gray", linestyle="--", label="Word onset")
        plt.title(f"{patient} {region}: LLH Diff ± 95% CI vs. Shift")
        plt.xlabel("Shift (ms)")
        plt.ylabel("LLH Diff")
        plt.tight_layout()
        plt.show()
else:
    print(f"No result CSVs found yet in {RESULTS_DIR}")